# A.R.I.D. MED-SAM YOLOv5s training

This notebook prepares the **Breeding Place Detection** dataset, converts its mixed polygon annotations to bounding boxes, trains YOLOv5s, evaluates the test split, and exports an ONNX model for the A.R.I.D. web dashboard.

Before running: create `MyDrive/ARID-YOLO/` in Google Drive and upload `Breeding Place Detection.zip` from `Z:\vistext-v3`. Then select **Runtime > Change runtime type > T4 GPU**, and run each cell in order.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
DRIVE_ROOT = Path('/content/drive/MyDrive/ARID-YOLO')
DATASET_ZIP = DRIVE_ROOT / 'Breeding Place Detection.zip'
WORK_ROOT = Path('/content/arid-medsam')
DATASET_DIR = WORK_ROOT / 'Breeding Place Detection'
RUNS_DIR = DRIVE_ROOT / 'runs'

assert DATASET_ZIP.exists(), f'Upload the dataset first: {DATASET_ZIP}'
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)
print('Dataset found:', DATASET_ZIP)

In [ ]:
!nvidia-smi

import torch
assert torch.cuda.is_available(), 'GPU is not enabled. Select Runtime > Change runtime type > T4 GPU.'
print('GPU ready:', torch.cuda.get_device_name(0))

In [ ]:
import shutil
import zipfile

if WORK_ROOT.exists():
    shutil.rmtree(WORK_ROOT)
WORK_ROOT.mkdir(parents=True)

with zipfile.ZipFile(DATASET_ZIP) as archive:
    archive.extractall(WORK_ROOT)

assert DATASET_DIR.exists(), f'Expected folder was not found: {DATASET_DIR}'
print('Extracted to:', DATASET_DIR)

In [ ]:
# Convert every polygon annotation to its enclosing YOLO bounding box.
converted = 0
box_rows = 0
empty_files = 0

for label_path in DATASET_DIR.glob('*/labels/*.txt'):
    source_lines = [line.strip() for line in label_path.read_text().splitlines() if line.strip()]
    if not source_lines:
        empty_files += 1
        continue

    output_lines = []
    for line in source_lines:
        parts = line.split()
        class_id = int(parts[0])
        values = [float(value) for value in parts[1:]]

        if len(values) == 4:
            x_center, y_center, width, height = values
            box_rows += 1
        elif len(values) >= 6 and len(values) % 2 == 0:
            xs = values[0::2]
            ys = values[1::2]
            x_min, x_max = min(xs), max(xs)
            y_min, y_max = min(ys), max(ys)
            x_center = (x_min + x_max) / 2
            y_center = (y_min + y_max) / 2
            width = x_max - x_min
            height = y_max - y_min
            converted += 1
        else:
            raise ValueError(f'Unsupported annotation in {label_path}: {line}')

        output_lines.append(
            f'{class_id} {x_center:.8f} {y_center:.8f} {width:.8f} {height:.8f}'
        )

    label_path.write_text('\n'.join(output_lines) + '\n')

print(f'Kept {box_rows} box annotations')
print(f'Converted {converted} polygon annotations to boxes')
print(f'Kept {empty_files} background label files')
assert converted == 161, f'Expected 161 polygon rows, found {converted}'

In [ ]:
# Write unambiguous paths for Colab/YOLOv5.
yaml_text = f'''path: {DATASET_DIR}
train: train/images
val: valid/images
test: test/images

nc: 5
names: ['Bottle', 'Coconut-Exocarp', 'Drain-Inlet', 'Tire', 'Vase']
'''
(DATASET_DIR / 'data.yaml').write_text(yaml_text)
print((DATASET_DIR / 'data.yaml').read_text())

In [ ]:
%cd /content
!rm -rf yolov5
!git clone --depth 1 https://github.com/ultralytics/yolov5.git
%cd /content/yolov5
!pip install -qr requirements.txt

In [ ]:
# Training checkpoints are saved to Drive so a completed epoch is not lost if Colab disconnects.
# subprocess with check=True stops here and shows the real training error instead of hiding it.
import subprocess

train_command = [
    'python', 'train.py',
    '--img', '640',
    '--batch', '16',
    '--epochs', '100',
    '--data', str(DATASET_DIR / 'data.yaml'),
    '--weights', 'yolov5s.pt',
    '--project', str(RUNS_DIR),
    '--name', 'medsam_yolov5s',
    '--exist-ok',
    '--workers', '2',
]
print('Starting training...')
subprocess.run(train_command, cwd='/content/yolov5', check=True)

In [ ]:
checkpoint_candidates = list(RUNS_DIR.rglob('best.pt'))
assert checkpoint_candidates, (
    f'No best.pt exists under {RUNS_DIR}. The training cell above failed or did not finish.'
)
BEST_PT = max(checkpoint_candidates, key=lambda path: path.stat().st_mtime)
print('Using checkpoint:', BEST_PT)

!python val.py --weights "{BEST_PT}" --data "{DATASET_DIR / 'data.yaml'}" --img 640 --task test
!python export.py --weights "{BEST_PT}" --include onnx --img 640 --opset 12

BEST_ONNX = BEST_PT.with_suffix('.onnx')
assert BEST_ONNX.exists(), f'ONNX export not found: {BEST_ONNX}'
MODEL_DIR = DRIVE_ROOT / 'models'
MODEL_DIR.mkdir(exist_ok=True)
FINAL_ONNX = MODEL_DIR / 'medsam_yolov5s.onnx'
shutil.copy2(BEST_ONNX, FINAL_ONNX)
print('Web model saved to:', FINAL_ONNX)

In [ ]:
from google.colab import files
files.download(str(FINAL_ONNX))